<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_slot.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.1 — Heat in a Stator Slot, by Finite Differences and by a PINN

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L7.1 · Fundamentals of PINNs**

A winding in the slot of an electrical machine carries a current and heats up.
How hot does it get inside, where no thermometer can go? You compute the answer
three ways — by finite differences, and by a physics-informed network with its
walls imposed first softly and then hard — and score all three against the
exact solution, which this slot happens to have.

### The running example

* **The slot.** 10 × 20 mm, in the stator of an electrical machine, its walls
  held at the temperature of the cooled iron, 90 °C.
* **The winding.** 32 round copper wires of 2 mm, in 4 columns of 8, all
  carrying the same current, 10 to 45 A. Each wire makes $RI^2$ of heat, and
  hot copper makes more of it.
* **The bundle** is copper, enamel and resin together, with an effective
  conductivity of 0.70 W/m·K — which is why the inside runs hot.

### Sections

| | what you do | TODO |
|---|---|---|
| **1** | the slot, and the heat its winding makes | the heat source |
| **2** | check that automatic differentiation is exact | — |
| **3** | the exact solution, as a sum of the slot's modes | — |
| **4** | the temperature by finite differences | — |
| **5** | the right mesh density for finite differences | the error of each grid |
| **6** | a PINN: collocation points, network size, soft and hard enforcement | the residual |
| **7** | the three answers compared | — |
| **8** | your report | your answers |
| **9** | mini project proposal | — |

The exercise form has three TODO cells, each with its answer in the comment
directly above the line to write. The light form has every cell written out.

---

## 0 · Setup

**What these cells do.** Fetch the three library files and import them.

**How.** On Colab the first cell downloads `course_core.py`, `pinn_core.py` and `problem.py` from the public course repository; the second imports them, fixes the random seed and sets the folder the report is written to.

In [ ]:
# files-cell v2 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository on every run, so a file left
# over from an earlier session cannot shadow it. Run this cell first.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    # on Colab always re-fetch: /content outlives a session and a stale copy
    # would silently shadow an updated one. Locally the repo's own files stay.
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
# a module imported before this cell would keep the previous file's contents
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

cc.OUTPUT_DIR = "Ex07.1_outputs"                          # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

**What you should see.** `device: cpu` on most machines and `dtype: torch.float64`.
Double precision is deliberate: a second derivative of a network is a
difference of differences, and in single precision the residual would be
dominated by rounding. No GPU is needed anywhere in this set.

---

## 1 · The slot and its heat

**What this cell does.** Prints the slot, its winding and the heat the winding makes at rated current.

**How.** `describe_slot()` reads the constants in `problem.py` and works out the resistance of one wire, its $RI^2$, and the heat of the whole winding. The problem, in the operator form of L7.1's **Operator Form** slide, is

$$k_{\text{eff}}\,\nabla^{2}\theta + q(\theta, I) = 0 \quad \text{in the slot}, \qquad \theta = 0 \ \text{ on the four walls},$$

where $\theta$ is the temperature rise above the wall. Every parameter is known and only the field is missing: a **forward problem**.

> In L7.1 $\theta$ is the network's parameters; here it is the temperature rise, the usual heat-transfer letter. Which is meant is always clear from what it is attached to.

In [ ]:
pb.describe_slot()

**What you should see.**

```
  slot            : 10 x 20 mm, walls held at 90 C
  winding         : 4 x 8 = 32 wires of 2 mm, copper fill 0.503
  bundle k_eff    : 0.70 W/m.K   (copper alone: about 400)
  current         : 31.4 A per wire = 10.0 A/mm^2   (the notebooks use 10 to 45 A)
  one wire at 90 C: R = 6.98 mOhm/m, R I^2 = 6.89 W/m
  whole slot      : 220 W per metre of slot
  heat source q   : 1.102 MW/m^3 at the wall temperature, +0.31 % per kelvin of rise
  thermal runaway : 158 A, 3.5 x the largest current used
```

![One stator slot, with half of each neighbour](https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/Ex07.1_slot.png)

**The winding is treated as one material.** Modelling every wire would make the
conductivity jump from about 400 W/m·K in the copper to about 0.2 in the enamel
and resin, at every wire edge. Machine designers average it instead: one
effective conductivity for the bundle, and the heat of the 32 wires spread
evenly over the slot. An alternating current heats like a direct one at its RMS
value, so $I$ here is the RMS current.

### The heat source

**What these cells do.** You write $q(\theta, I)$, the heat the winding makes per cubic metre when the slot is $\theta$ kelvin above the wall; the next cell checks it against `pb.heat_source` and plots the winding's heat against the current at two copper temperatures.

**How.** One metre of wire has resistance $\rho/A_{\text{wire}}$ and makes $RI^2$ of heat. Thirty-two of them, spread over the slot's cross-section, make

$$q = \text{fill}\cdot\rho(T)\,J^2, \qquad J = \frac{I}{A_{\text{wire}}}, \qquad \rho(T) = \rho_{20}\,[1 + \alpha\,(T - 20)], \qquad T = 90 + \theta,$$

with `pb.FILL`, `pb.A_WIRE`, `pb.RHO_CU`, `pb.ALPHA_CU` and `pb.T_WALL` for the constants.

In [ ]:
# TODO 1: write the heat source q(theta, I), in W/m^3.
#
#   def heat_source(theta, I):
#       J = I / pb.A_WIRE                                                # A/m^2
#       rho = pb.RHO_CU * (1 + pb.ALPHA_CU * (pb.T_WALL + theta - 20))   # at T = 90 C + theta
#       return pb.FILL * rho * J ** 2                                    # 32 wires, over the slot
#
# The copper is at the wall temperature PLUS the rise. Leave out pb.T_WALL and
# its resistance is worked out at theta degrees - some ninety kelvin too cold.

raise NotImplementedError("Write heat_source(theta, I)")

In [ ]:
theta_test = np.array([0.0, 10.0, 40.0])        # kelvin above the wall
for I in (pb.I_RANGE[0], pb.I_RATED, pb.I_RANGE[1]):
    check(f"heat_source at {I:.1f} A", heat_source(theta_test, I) / pb.heat_source(theta_test, I),
          np.ones(3), tol=1e-12)

currents = np.linspace(*pb.I_RANGE, 60)
fig, ax = plt.subplots(figsize=(6.6, 4.0))
for T, col in ((pb.T_WALL, CYCLE[1]), (pb.T_WALL + 40, CYCLE[0])):
    ax.plot(currents, pb.N_WIRES * pb.heat_per_wire(currents, T), color=col, lw=2,
            label=f"copper at {T:.0f} °C")
ax.axvline(pb.I_RATED, color="0.6", ls=":", lw=1)
ax.set_xlabel("current per wire  [A]"); ax.set_ylabel("heat  [W per metre of slot]")
ax.set_title("RI² of the whole winding"); ax.legend(frameon=False)
plt.show()

**What you should see.** Three `PASS` lines, and two rising curves. Both grow
as $I^2$; the upper one is the same winding 40 K hotter and makes **12 % more
heat**, because copper's resistance has risen by that much. Inside the slot the
copper is hotter than the wall, so it makes more heat, which heats it further:
a positive feedback. Split the heat into a fixed part and a part that grows
with the rise, $q = c_0 + c_1\theta$ — `pb.heat_coefficients(I)` returns the
two. From here on the notebook uses `pb.heat_source`, the function your
version was just checked against.

---

## 2 · Automatic differentiation is exact

**What this cell does.** Checks that autograd's second derivatives agree with the exact ones, on the nodes of the finite-difference grid used in sections 4 and 7, and on a temperature field of the slot's own shape. It also introduces the one number that decides how strongly hot copper pushes the temperature up.

**Why this field.** Any function whose derivatives are known exactly would test autograd. We use a dome of the slot's shape, zero on all four walls and 1 in the middle, because it comes back in section 3:

$$u(x, y) = \cos\frac{\pi x}{W}\,\cos\frac{\pi y}{H}, \qquad W = 10\ \text{mm},\ H = 20\ \text{mm}.$$

Differentiating twice by hand, $u_{xx} = -(\pi/W)^2 u$ and $u_{yy} = -(\pi/H)^2 u$, so

$$\nabla^2 u = -\lambda_1\,u, \qquad \lambda_1 = \left(\frac{\pi}{W}\right)^2 + \left(\frac{\pi}{H}\right)^2 = 0.0987 + 0.0247 = 0.1234\ \text{mm}^{-2}.$$

**Why $\lambda_1$ is called an eigenvalue.** $\nabla^2$ turns the dome back into the dome, multiplied by a number — as a matrix turns an eigenvector $v$ into $\lambda v$. $\lambda_1$ is the slot's smallest eigenvalue: the dome is the pattern that loses heat to the walls most slowly, $k_{\text{eff}}\lambda_1$ watts per cubic metre for every kelvin of its height.

**Why it matters.** Hot copper adds $c_1$ watts per cubic metre for every kelvin of rise. The dome gains $c_1$ and loses $k_{\text{eff}}\lambda_1$ per kelvin, so its height is amplified by

$$\frac{1}{1 - c_1/(k_{\text{eff}}\,\lambda_1)}.$$

Where $c_1$ reaches $k_{\text{eff}}\lambda_1$ the factor is infinite and there is no steady temperature: **thermal runaway**.

In [ ]:
# the dome on the interior nodes of the 41 x 81 finite-difference grid
W, H = 2 * pb.A_HALF, 2 * pb.B_HALF                        # slot width and depth, m
lam_x, lam_y = (np.pi / W) ** 2, (np.pi / H) ** 2           # the two terms of lambda_1, 1/m^2
lam1 = lam_x + lam_y                                        # the slot's first eigenvalue, 1/m^2
xg, yg = pb.fdm_grid(41)                                    # the nodes of sections 4 and 7
Xg, Yg = np.meshgrid(xg[1:-1], yg[1:-1])                    # interior nodes only
xy = to_tensor(np.stack([Xg.ravel(), Yg.ravel()], axis=1), requires_grad=True)  # differentiable coordinates
u = torch.cos(np.pi * xy[:, :1] / W) * torch.cos(np.pi * xy[:, 1:] / H)

u_xx = d2(u, xy, 0)  # second derivative in x, by autograd
u_yy = d2(u, xy, 1)  # second derivative in y, by autograd

# autograd against the exact derivatives, compared per unit of lambda_1
check("u_xx = -(pi/W)^2 u", to_numpy(u_xx) / lam1, -lam_x * to_numpy(u) / lam1, tol=1e-9)
check("u_yy = -(pi/H)^2 u", to_numpy(u_yy) / lam1, -lam_y * to_numpy(u) / lam1, tol=1e-9)
print(f"\n{len(xy)} nodes;  lambda_1 = {lam_x/1e6:.4f} + {lam_y/1e6:.4f} = {lam1/1e6:.4f} per mm^2")

for I in (pb.I_RATED, pb.I_RANGE[1], pb.runaway_current()):
    c0, c1 = pb.heat_coefficients(I)                        # q = c0 + c1 theta, W/m^3
    ratio = c1 / (pb.K_EFF * lam1)                          # gain over loss, per kelvin
    amp = f"x{1 / (1 - ratio):.3f}" if ratio < 0.999 else "infinite: thermal runaway"
    print(f"  I = {I:6.1f} A: hot copper amplifies the dome {amp}")

# the dome drawn over the slot, 1 in the middle and 0 on the walls
fig = plt.figure(figsize=(10.0, 4.6))
ax3 = fig.add_subplot(1, 2, 1, projection="3d")
Xs, Ys = np.meshgrid(xg, yg)
ax3.plot_surface(Xs * 1e3, Ys * 1e3, np.cos(np.pi * Xs / W) * np.cos(np.pi * Ys / H),
                 cmap="inferno", linewidth=0)
ax3.set_xlabel("x  [mm]"); ax3.set_ylabel("y  [mm]"); ax3.set_zlabel("u")
ax3.set_box_aspect((1, 2, 1)); ax3.set_title("the dome u, over the 10 x 20 mm slot")
ax2 = fig.add_subplot(1, 2, 2)
pb.plot_field((np.cos(np.pi * Xs / W) * np.cos(np.pi * Ys / H)).ravel(), 41, 81, ax=ax2,
              wires=True, title="from above", label="u")
plt.tight_layout(); plt.show()

**What you should see.** Two `PASS` lines with errors around $10^{-16}$ —
autograd's derivatives are exact up to the rounding of double precision, not an
approximation like a finite difference — on the 3,081 interior nodes of the
grid; $\lambda_1 = 0.1234$ per mm²; an amplification of ×1.041 at rated
current, ×1.088 at 45 A, and thermal runaway at 158 A, three and a half times
the largest current the set uses. The figure is the dome over the slot.

---

## 3 · The exact solution

**What this cell does.** Evaluates the exact temperature rise at rated current, and shows how it is built from domes.

**How.** Because the heat is $q = c_0 + c_1\theta$ with constant $c_0$ and $c_1$, and the slot is a rectangle held at zero, the equation $k_{\text{eff}}\nabla^2\theta + c_1\theta = -c_0$ can be solved exactly. Every mode of the slot, $u_{mn} = \sin\frac{m\pi x'}{W}\sin\frac{n\pi y'}{H}$ with $x' = x + W/2$ and $y' = y + H/2$, is zero on the walls and has $\nabla^2 u_{mn} = -\lambda_{mn}u_{mn}$ with $\lambda_{mn} = (m\pi/W)^2 + (n\pi/H)^2$. A constant is $\sum 16/(\pi^2 mn)\,u_{mn}$ over odd $m$ and $n$, so

$$\theta = \sum_{m,n\ \text{odd}} \frac{16\,c_0}{\pi^2\,m\,n\,\bigl(k_{\text{eff}}\lambda_{mn} - c_1\bigr)}\;u_{mn}.$$

The first term, $m = n = 1$, is the dome of section 2, 21.5 K high; the smaller domes after it pull the middle down. `pb.exact_solution(I, x, y)` sums the terms up to 399 in each direction, which is exact to about $10^{-5}$ K. It is the reference both methods are scored against. A real slot — a rounded bottom, tooth tips, a liner — has no such formula, which is why finite differences and networks exist.

In [ ]:
x41, y41 = pb.fdm_grid(41)                                  # the 41 x 81 nodes
theta_exact = pb.exact_solution(pb.I_RATED, x41, y41)       # 81 x 41, kelvin
print(f"exact hot spot at rated current: {theta_exact.max():.5f} K above the wall")
for n in (1, 3, 9, 49, 399):                                # terms in each direction
    centre = pb.exact_solution(pb.I_RATED, [0.0], [0.0], n_terms=n)[0, 0]
    print(f"  terms up to {n:3d}: hot spot {centre:8.4f} K")

**What you should see.** An exact hot spot of **18.74033 K**, 108.7 °C in the
copper. One dome gives 21.54 K, too hot; with terms up to 3 the sum swings
below, to 18.17 K; up to 9 it is within 0.05 K, up to 49 within a thousandth,
and then it settles. The terms alternate in sign at the centre, so the sum
closes in from both sides.

---

## 4 · The temperature by finite differences

**What this cell does.** Solves the slot at rated current on a 41 × 81 grid, compares the hot spot with the exact one, and draws the mesh beside the temperature it produced.

**How.** The grid has 41 nodes across the 10 mm width and $2 \times 41 - 1 = 81$ along the 20 mm depth, so the spacing is the same both ways, $h = 10/40 = 0.25$ mm. The wall nodes are $\theta = 0$ and are not unknowns; each of the $39 \times 79 = 3081$ nodes inside is one unknown. At every one of them the five-point stencil stands in for the Laplacian,

$$k_{\text{eff}}\,\frac{\theta_N + \theta_S + \theta_E + \theta_W - 4\theta_P}{h^2} + c_0 + c_1\theta_P = 0,$$

so every node gives one linear equation and the slot is one sparse system. `pb.fdm_solve` builds and solves it. Why 41 nodes is section 5.

In [ ]:
NX = 41                                                     # nodes across; section 5 shows why
X, Y, theta = pb.fdm_solve(pb.I_RATED, NX)                  # 81 x 41 nodes, walls included
print(f"hot spot {theta.max():.5f} K  (exact {theta_exact.max():.5f} K), "
      f"worst error {np.abs(theta - theta_exact).max():.1e} K")
print(f"{(NX - 2) * (2 * NX - 3):,} unknowns, one linear system")

# the mesh as it is: every grid line, the wires, and one node's stencil
fig, axes = plt.subplots(1, 2, figsize=(10.5, 6.4))
ax = axes[0]
x_mm, y_mm = x41 * 1e3, y41 * 1e3
for xv in x_mm:
    ax.plot([xv, xv], [y_mm[0], y_mm[-1]], color="0.7", lw=0.4)
for yv in y_mm:
    ax.plot([x_mm[0], x_mm[-1]], [yv, yv], color="0.7", lw=0.4)
ax.add_patch(plt.Rectangle((x_mm[0], y_mm[0]), 10, 20, fill=False, ec=CYCLE[1], lw=2.5))
for i in range(pb.N_COLS):                                  # the 32 wires, for orientation
    for j in range(pb.N_ROWS):
        ax.add_patch(plt.Circle((-5 + (i + 0.5) * 10 / pb.N_COLS, -10 + (j + 0.5) * 20 / pb.N_ROWS),
                                1.0, color=CYCLE[0], alpha=0.12))
ci, cj = 14, 52                                             # one interior node
for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
    ax.plot([x_mm[ci], x_mm[ci + di]], [y_mm[cj], y_mm[cj + dj]], color=CYCLE[2], lw=2)
    ax.plot(x_mm[ci + di], y_mm[cj + dj], "o", ms=5, color=CYCLE[2])
ax.plot(x_mm[ci], y_mm[cj], "o", ms=7, color=CYCLE[0])
ax.set_aspect("equal"); ax.set_xlabel("x  [mm]"); ax.set_ylabel("y  [mm]")
ax.set_title(f"the mesh: {NX} x {2*NX - 1} nodes, h = {(x41[1]-x41[0])*1e3:.2f} mm")
pb.plot_field(theta.ravel(), NX, 2 * NX - 1, ax=axes[1], wires=True,
              title="temperature rise at rated current")
plt.tight_layout(); plt.show()

**What you should see.** A hot spot of **18.7377 K** against the exact 18.7403,
a worst error of about 0.004 K anywhere in the slot, from 3,081 unknowns.

**The left panel is the mesh itself**, every line drawn, 0.25 mm apart: 41
lines across and 81 along. Every crossing inside the blue wall is an unknown;
one of them is picked out with the four neighbours its equation uses, and every
other has the same cross. The mesh does not follow the wires: the winding is
one material with its heat spread evenly, so a node inside a wire, on its edge
or between two wires gets the same equation. The circles only show where the
copper is. The field on the right is symmetric, hottest in the middle, zero on
the walls.

---

## 5 · The right mesh density

**What this cell does.** Solves the slot on six grids, from 11 to 321 nodes across, each with half the spacing of the one before, times every solve, and — in the TODO below — measures each against the exact solution.

**How.** For each grid it solves once to warm up, then three more times, and keeps the middle time, building the grid and the matrix included. The exact solution is evaluated on each grid's own nodes.

In [ ]:
GRIDS = [11, 21, 41, 81, 161, 321]
sols, times = {}, {}
for nx in GRIDS:
    pb.fdm_solve(pb.I_RATED, nx)                             # warm-up, not timed
    ts = []
    for _ in range(3):
        t0 = time.perf_counter()
        _, _, th = pb.fdm_solve(pb.I_RATED, nx)              # grid + matrix + solve
        ts.append(time.perf_counter() - t0)
    sols[nx], times[nx] = th, float(np.median(ts))
print("solved on", ", ".join(f"{g} x {2*g-1}" for g in GRIDS))

In [ ]:
# TODO 2: the worst error of each grid, against the exact solution on its nodes.
#
#   errs = {}
#   for nx in GRIDS:
#       x, y = pb.fdm_grid(nx)                                   # this grid's nodes
#       errs[nx] = float(np.abs(sols[nx] - pb.exact_solution(pb.I_RATED, x, y)).max())   # worst error, K

raise NotImplementedError("Write the errs dictionary")

In [ ]:
TARGET = 0.01                                               # K: finer than any decision about the winding
hs = np.array([10.0 / (g - 1) for g in GRIDS])              # node spacing, mm
hot = np.array([sols[g].max() for g in GRIDS])
err = np.array([errs[g] for g in GRIDS])
for g, h, p, e in zip(GRIDS, hs, hot, err):
    print(f"  {g:4d} x {2*g-1:4d}  h = {h:6.4f} mm  hot spot {p:9.5f} K  worst error {e:.1e} K"
          f"  {times[g]*1e3:7.1f} ms" + ("   <- used" if g == NX else ""))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.0, 4.2))
a1.semilogx(hs, hot, "o-", color=CYCLE[0], lw=2, label="finite differences")
a1.axhline(theta_exact.max(), color="0.3", ls="--", lw=1, label="exact")
a1.set_xlabel("node spacing $h$  [mm]"); a1.set_ylabel("hot spot  [K]")
a1.set_title("the hot spot as the mesh is refined"); a1.legend(frameon=False)
a2.loglog(hs, err, "o-", color=CYCLE[0], lw=2)
a2.axhline(TARGET, color=CYCLE[3], ls="--", lw=1.2)
a2.text(hs[-1], TARGET * 1.3, "0.01 K", color=CYCLE[3], fontsize=9)
a2.set_xlabel("node spacing $h$  [mm]"); a2.set_ylabel("worst error  [K]")
a2.set_title("the worst error as the mesh is refined")
k = GRIDS.index(NX)
for ax, yv in ((a1, hot), (a2, err)):
    ax.plot(hs[k], yv[k], "o", ms=13, mfc="none", mec=CYCLE[1], mew=2)   # the grid used
plt.tight_layout(); plt.show()

**What you should see.** On the left, the hot spot climbing towards the exact
value as $h$ shrinks: the coarsest grid is 0.04 K too cold, the finest agrees to
about $10^{-4}$ K. On the right, the worst error falling four times each time
the spacing halves — the stencil's error is proportional to $h^2$ — while the
time grows five to six times.

**Why 41 × 81.** A hundredth of a kelvin, the dashed line, is finer than any
decision about the winding needs: insulation classes are set in whole kelvin.
The 21-node grid misses it (0.015 K); the 41-node grid is the coarsest that
meets it, with 0.004 K in a few milliseconds; every finer grid buys accuracy
nothing here uses, at five to six times the time per halving. So 41 × 81 it is — and
3,081 unknowns is also the number the network of section 6 is measured against.

---

## 6 · A physics-informed network

**What this cell does.** Builds the network, chooses its collocation points, and prints its size against the number of points it trains on.

**How.** The network sees three numbers: $\xi = x/a$ and $\eta = y/b$, which run from −1 to 1 across the slot, and $s$, the current scaled to the same range — so **one training answers every current** from 10 to 45 A. Its output is scaled to the size of the answer, $\hat\theta = 20\ \text{K}\,(I/I_{\text{rated}})^2\,N(\xi, \eta, s)$, so it only has to learn the shape. Two ways to impose the walls, L7.1's **Soft Enforcement** and **Hard Enforcement** slides:

* **soft enforcement** — a penalty in the loss, asked for at points on the walls;
* **hard enforcement** — the output multiplied by the mask $(1 - \xi^2)(1 - \eta^2)$, zero on every wall.

**The collocation points, and the network's size.** The residual is evaluated at 2000 points inside the slot and the current range, by Latin hypercube, and the soft penalty at 400 points on the walls. Finite differences needed 3,081 unknowns for **one** current; the network has about as many parameters — four layers of 32 — and the 2000 points cover every current at once. L7.1's **Solution Ansatz** slide asks for at least $N^*$ collocation points, a number set by the neurons: `describe` prints both. The points stay fixed, because L-BFGS needs the same points at every step.

In [ ]:
a, b = pb.A_HALF, pb.B_HALF
I_LO, I_HI = pb.I_RANGE
S = 20.0                                                    # K: the size of the answer
Q = pb.heat_coefficients(pb.I_RATED)[0]                     # W/m^3: the size of the heat

def current(s):
    return I_LO + (s + 1) / 2 * (I_HI - I_LO)               # s in -1..1 back to amperes

class Slot(torch.nn.Module):                                # a network: layers in __init__, the computation in forward
    def __init__(self, hard, width=32, layers=4):
        super().__init__()
        self.hard = hard
        self.net = MLP(n_in=3, n_hidden=width, n_layers=layers)   # dense tanh network: xi, eta, s in

    def forward(self, p):
        xi, eta, s = p[:, :1], p[:, 1:2], p[:, 2:]
        out = S * (current(s) / pb.I_RATED) ** 2 * self.net(p)
        return out * (1 - xi ** 2) * (1 - eta ** 2) if self.hard else out   # the mask, for hard enforcement

box = ((-1, 1), (-1, 1), (-1, 1))                           # xi, eta and the current
inner = to_tensor(interior_points(2000, box, method="lhs", seed=1), requires_grad=True)  # collocation points, differentiable
w2 = boundary_points(100, ((-1, 1), (-1, 1)), seed=1)       # 400 points on the four walls
walls = to_tensor(np.hstack([w2, np.random.default_rng(1).uniform(-1, 1, (len(w2), 1))]))  # each at a random current

describe(Slot(hard=True).net, n_collocation=len(inner))  # print the parameter count and the collocation-to-parameter ratio
fig, ax = plt.subplots(figsize=(4.2, 6.0))
pin = to_numpy(inner)
ax.plot(pin[:, 0] * a * 1e3, pin[:, 1] * b * 1e3, ".", ms=2.5, color=CYCLE[1], label="inside: the residual")
ax.plot(w2[:, 0] * a * 1e3, w2[:, 1] * b * 1e3, ".", ms=4, color=CYCLE[0], label="walls: the soft penalty")
ax.set_aspect("equal"); ax.set_xlabel("x  [mm]"); ax.set_ylabel("y  [mm]")
ax.set_title("collocation points, all currents together")
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1))
plt.show()

**What you should see.** The network's size and its collocation ratio, and the
slot filled with blue points and ringed in orange. Unlike the mesh of section
4 the points follow no grid: they are drawn at random, spread evenly by the
Latin hypercube, and each one also carries its own current.

### The residual

**What this cell does.** You write the residual: the slot's equation with the network in place of $\theta$, divided by the rated heat $Q$ so that it is a number of order one.

**How.** `d2(th, p, 0)` is the second derivative in $\xi$. Since $\xi = x/a$, $\partial^2/\partial x^2 = (1/a^2)\,\partial^2/\partial\xi^2$, and likewise in $\eta$ with $b$. The heat is `pb.heat_source` at the current `current(p[:, 2:])`.

In [ ]:
# TODO 3: write the residual, scaled by the rated heat Q.
#
#   def residual(model, p):
#       th = model(p)                                                     # K
#       lap = d2(th, p, 0) / a ** 2 + d2(th, p, 1) / b ** 2               # the Laplacian by autograd, in metres
#       return (pb.K_EFF * lap + pb.heat_source(th, current(p[:, 2:]))) / Q
#
# Leave out the 1/a^2 and 1/b^2 and the network solves a slot 2 m by 2 m.

raise NotImplementedError("Write residual(model, p)")

### Training, with soft and with hard enforcement

**What this cell does.** Trains the same network twice, on the same points with the same budget: once with soft enforcement of the walls, once with hard.

**How.** The soft loss is the mean squared residual plus the mean squared wall rise, $\mathcal{L} = \overline{r^2} + w\,\overline{(\hat\theta_{\text{wall}}/S)^2}$ with $w = 1$; the hard loss is the residual alone, because the mask has already met the walls. Both use the course's schedule: 1500 steps of Adam at a learning rate of $2\times10^{-3}$, then 150 steps of L-BFGS. About a minute each on a CPU.

In [ ]:
nets, train_time = {}, {}
for name, hard in (("soft", False), ("hard", True)):
    set_seed(88)  # fix the random start so a run repeats
    model = Slot(hard).to(DEVICE)                            # four layers of 32

    def loss():                                              # what train_two_stage minimises
        l = residual(model, inner).pow(2).mean()             # the equation, inside
        if not hard:
            l = l + (model(walls) / S).pow(2).mean()         # the soft wall penalty, w = 1
        return l

    t0 = time.perf_counter()
    train_two_stage(model, loss, adam_steps=1500, lbfgs_steps=150, lr=2e-3, report_every=0)  # Adam first, then L-BFGS
    nets[name], train_time[name] = model, time.perf_counter() - t0
    print(f"{name} enforcement: trained in {train_time[name]:.0f} s")

**What you should see.** Two lines, each a minute or so.

---

## 7 · The three answers compared

**What this cell does.** Scores finite differences on the 41 × 81 grid, and the networks with soft and with hard enforcement on the same nodes, against the exact solution at six currents from 10 to 45 A. It prints the worst error of each, the time one current takes, and the training time, then maps each method's error at 45 A.

**How.** For each current the exact solution and the finite-difference solution are computed on the grid's nodes, and each network is evaluated there. The maps show the error with its sign: blue where the method is too cold, red where it is too hot, white where it is right.

In [ ]:
TEST_I = np.array([10.0, 17.5, 25.0, pb.I_RATED, 38.0, 45.0])
P = np.stack([np.meshgrid(x41, y41)[0].ravel(), np.meshgrid(x41, y41)[1].ravel()], axis=1)

def evaluate(model, I):
    s = 2 * (I - I_LO) / (I_HI - I_LO) - 1
    p = to_tensor(np.hstack([P / [a, b], np.full((len(P), 1), s)]))  # the network's inputs on the grid
    with torch.no_grad():  # no gradient tracking: evaluation only
        return to_numpy(model(p)).reshape(len(y41), len(x41))

def middle(fn, reps=5):
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

exact = {I: pb.exact_solution(I, x41, y41) for I in TEST_I}
methods = {"finite differences": lambda I: pb.fdm_solve(I, NX)[2],
           "PINN, soft enforcement": lambda I: evaluate(nets["soft"], I),
           "PINN, hard enforcement": lambda I: evaluate(nets["hard"], I)}
worst, per_current = {}, {}
for name, f in methods.items():
    worst[name] = max(np.abs(f(I) - exact[I]).max() for I in TEST_I)
    per_current[name] = middle(lambda: f(28.0))
train = {"finite differences": 0.0, "PINN, soft enforcement": train_time["soft"],
         "PINN, hard enforcement": train_time["hard"]}

print(f"{'':26s}{'worst error':>13s}{'per current':>14s}{'training':>11s}")
for name in methods:
    print(f"  {name:24s}{worst[name]:11.4f} K{per_current[name]*1e3:11.2f} ms{train[name]:9.0f} s")
saved_ms = per_current["finite differences"] - per_current["PINN, hard enforcement"]
if saved_ms > 0:
    print(f"\nthe hard-enforcement network is {per_current['finite differences'] / per_current['PINN, hard enforcement']:.0f} x "
          f"faster per current; its training pays after about {train_time['hard'] / saved_ms:,.0f} currents")

fig, axes = plt.subplots(1, 3, figsize=(12.0, 5.6))
for ax, (name, f) in zip(axes, methods.items()):
    pb.plot_field((f(45.0) - exact[45.0]).ravel(), NX, 2 * NX - 1, ax=ax,
                  title=f"{name}\nminus exact, 45 A", label="error  [K]", cmap="coolwarm")
plt.tight_layout()
plt.savefig(cc.output_path("Ex07.1_report.png"), dpi=150, bbox_inches="tight")
plt.show()

**What you should see.** Finite differences within about **0.007 K** at every
current, in a few milliseconds each. The network with **hard enforcement**
within about **0.02 K**, and exactly zero on the walls. The network with **soft
enforcement** up to about **1.5 K** off — worst at the walls, where the
condition was only asked for, and spreading inward from there.

**What the comparison says.**

* **Hard enforcement is the better of the two networks**, by almost a hundred
  times. A soft wall is a request: the loss trades it against the
  equation, the wall ends up a fraction of a kelvin warm, and because the walls
  set the temperature inside, the error spreads through the whole slot. The
  mask never offers that trade.
* **For one slot and one current, finite differences are more accurate and
  cheaper**: the solve takes milliseconds and needs no training.
* **The network's advantage is that one training answers every current.** It
  evaluates a current faster than finite differences solve one, and repays its
  minute of training only after thousands of currents — a design sweep, a
  controller asking every few milliseconds, a digital twin. Where a mesh is
  costly — awkward 3-D shapes, unknown parameters found from data — the balance
  moves further, and that is where the rest of Part 2 goes.

---

## 8 · Your report

**What these cells do.** You answer four questions and the question that concludes the set; the next cell writes `Ex07.1_report.md` with the results table, the error maps of section 7 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L7.1's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "Finite differences turned the slot into one linear system. What are its unknowns, "
    "how many did the 41 x 81 grid have, and what takes the place of the mesh in a "
    "physics-informed network? (-> L7.1 Q4)": """
""",
    "Why is 41 x 81 the right mesh for this slot, and how did the error change each "
    "time the spacing h was halved? (-> L7.1 Q4)": """
""",
    "Why is a wall imposed by soft enforcement met only approximately, and why does "
    "its error not stay on the wall? (-> L7.1 Q7)": """
""",
    "What does the mask of hard enforcement guarantee, and what would it cost for a "
    "slot with a rounded bottom? (-> L7.1 Q9)": """
""",
    "To conclude: finite differences won this slot. For which problem would you train "
    "a physics-informed network instead, and what would you check it against? "
    "(-> L7.1, Exercise slide)": """
""",
}

In [ ]:
missing = [q for q, text in ANSWERS.items() if len(text.split()) < 8]
if NAME == "Your Name" or missing:
    print("Not ready:", "your name," if NAME == "Your Name" else "", len(missing), "answer(s) still empty")
else:
    rows = ["| method | worst error | per current | training |", "|---|---|---|---|"]
    rows += [f"| {n} | {worst[n]:.4f} K | {per_current[n]*1e3:.2f} ms | {train[n]:.0f} s |" for n in methods]
    out = ["# Ex_07.1 — Heat in a stator slot", "", f"**{NAME}**", "",
           "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", f"Exact hot spot at rated current: {theta_exact.max():.4f} K above the wall.", "",
           *rows, ""]
    for q, text in ANSWERS.items():
        out += [f"## {q}", "", text.strip(), ""]
    path = cc.output_path("Ex07.1_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready:` until every answer is yours, then the
path of the report. Then run the cell below: it turns the report into a PDF,
with the error maps of section 7, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex07.1_report.md into Ex07.1_report.pdf, with any figure
# saved as Ex07.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex07.1_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex07.1_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex07.1_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex07.1_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**: the course builds it with a classical
solver once you have chosen the project, and shows that it has converged. You
do not solve the physics a second way; the month goes into the deep learning —
making a network predict what the ground truth says, and showing how well it
does.

### Mini project A · The slot with its insulation

**The problem.** Ex_07.1 averaged the winding into one material. Here every one
of the 32 wires is resolved: a 2 mm copper core, its enamel coat, and the
impregnating resin that fills the gaps between the wires. The enamel is thin:
a grade 2 wire, the usual choice for a motor, is at least 0.075 mm thicker than
its 2 mm conductor (IEC 60317), so the coat is about 0.04 mm — a fiftieth of
the wire. Copper conducts about 400 W/m·K, the enamel and an unfilled resin
about 0.2, so the conductivity drops some two-thousandfold at every copper
edge. The heat is made in the copper only.

**Why a plain PINN is not enough.** The temperature is continuous at every
edge but its slope is not — the heat flow $k\,\partial\theta/\partial n$ is what
stays the same on both sides. A smooth network rounds that corner off, and a
coat 0.04 mm thick in a 10 mm slot is so thin that random collocation points
hardly ever land in it.

**What you build.** A domain-decomposed PINN: one network per material, joined
by interface terms in the loss that ask for equal temperature and equal heat
flow across every edge. You choose how to sample the thin layers, how to weight
the interface terms, and whether the walls use soft or hard enforcement.

**The ground truth you get.** The resolved slot solved by finite differences,
with several nodes across the enamel and a conductivity set cell by cell, for
the currents of Ex_07.1, with its grid-refinement table.

**What you hand in.** Error maps against the ground truth, in each material;
the hot spot and its error; how well the heat flow matches across the edges;
and how far the averaged slot of Ex_07.1 was from the resolved one.

### Mini project B · How old is the insulation?

**The problem.** Insulation ages. Heat degrades the enamel and the resin, the
resin comes away from the wires and voids open. A void holds air, which
conducts about a tenth as well as the resin it replaces, so the conductivity
falls where that happens — the copper runs hotter, which ages the insulation
faster. Nobody can open a machine in service to look. A few temperature sensors
in the slot, read at a few currents, are all there is.

**What you build.** An inverse PINN on the averaged slot of Ex_07.1, now with a
conductivity that varies over the slot: one network for the temperature
$\theta(x, y, I)$ and one for the conductivity $k(x, y)$, kept positive, fitted
together to the heat equation $\nabla\cdot(k\nabla\theta) + q = 0$ and to the
sensor readings.

**The ground truth you get.** Slots with a degraded region hidden in them,
solved by finite differences: the sensor readings, with their positions,
currents and noise level, and the true conductivity and temperature fields to
score yourself against afterwards.

**What you hand in.** The recovered conductivity against the true one, the
predicted hot spot against the true one, and how both depend on the number of
sensors and the noise.